<a href="https://colab.research.google.com/github/zhang-dny/walmart-hurricane-project/blob/main/Midterm_teamRamen_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 Midterm Project: Walmart Hurricane Analytics

**Team Ramen:** Antony Lin (`antonylin0428`) and Danny (`zhang-dny`)  
**Repo:** https://github.com/zhang-dny/walmart-hurricane-project  
**Storm:** Hurricane Helene, landfall near Perry, FL late on Sept 26, 2024 (about 11:10 pm Eastern)

We are Walmart's data science consultants. This notebook cleans five messy data files, adds real weather and Helene's real storm track, answers the five project questions, and ends with a recommendation and our reflection.

**Results at a glance**

| Question | Answer |
|---|---|
| Q1 Demand surge | Orders ran **2.4x** normal in the 7 days before landfall (Sept 19-25 vs Sept 1-15). Breakfast (all Pop-Tarts) rose most, **+414%**; Emergency supplies added the most units |
| Q2 SKU consolidation | **45 SKUs became 20 products.** Pop-Tarts went from the best single SKU at #8 of 45 to the #2 product of 20, 3.8x bigger |
| Q3 Weather | Orders lead Orlando's peak wind by **2 days** (r = 0.84, n = 28 days). Same-day correlation is only about 0.4 |
| Q4 Stores | **No geographic pattern**: all 12 stores surged 2.2x-2.5x and none is an outlier |
| Q5 Board games | The +250% is a **peak-days** figure. Monopoly-type board games did rise more than the basket, but playing cards in the same category did not. Recommendation: stock extra board games, sized like the basket and behind Pop-Tarts and Emergency supplies |

**How to run:** open in Google Colab, choose *Runtime > Restart session and run all*. It clones the repo, needs internet for the weather call (about one minute in total), and prints which weather source it used.

**Data and sources**
- The five supplied files in `data/`: `walmart_transactions.csv`, `store_locations.csv`, `product_catalog.csv`, `distribution_centers.csv`, `inventory_and_sales.db`.
- Weather: Open-Meteo historical archive API, base URL `https://archive-api.open-meteo.com/v1/archive` (free, no key), Orlando, daily values in America/New_York.
- Storm track: NOAA National Hurricane Center best track for Helene (AL092024), saved as `data/helene_best_track.csv`. Source: `https://ftp.nhc.noaa.gov/atcf/archive/2024/bal092024.dat.gz`.

**Sections:** 1 Cleaning, 2 Q1, 3 Q2, 4 Q3, 5 Q4, 6 Q5, 7 Recommendations, 8 Decision log, 9 Reflection, 10 Final self-check.

## 1. Data loading and cleaning pipeline

One pipeline cleans every file, and every decision goes in one decision log (`File | Issue | Decision | Rows affected | Reason`). Each cleaning step has an `assert` so a bad fix fails loudly instead of quietly changing the numbers.

In [ ]:
import subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Find the team repo: Colab clones it, a local copy is used if one exists.
REPO_URL = "https://github.com/zhang-dny/walmart-hurricane-project.git"
candidates = [Path("/content/walmart-hurricane-project"), Path.cwd(), Path.cwd().parent,
              Path("/home/claude/walmart-hurricane-project")]
REPO_PATH = next((p for p in candidates if (p / "data" / "walmart_transactions.csv").exists()), None)
if REPO_PATH is None:
    REPO_PATH = Path("/content/walmart-hurricane-project")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
DATA = REPO_PATH / "data"
print("Using repo:", REPO_PATH.name)

LANDFALL = pd.Timestamp("2024-09-26")      # Helene landfall (Florida Big Bend), per the project notes
message_charts = []                         # every chart with a message title adds its title here

In [ ]:
tx_raw = pd.read_csv(DATA / "walmart_transactions.csv")
stores_raw = pd.read_csv(DATA / "store_locations.csv")

MISSING_TOKENS = ["", "NULL", "null", "N/A", "n/a", "NaN", "nan", "None"]

CATEGORY_MAP = {
    "emergency": "Emergency", "emergency supplies": "Emergency",
    "grocery": "Grocery", "groceries": "Grocery",
    "electronics": "Electronics",
    "entertainment": "Entertainment", "games & entertainment": "Entertainment",
    "household": "Household", "household essentials": "Household",
    "baby": "Baby", "baby products": "Baby",
    "breakfast": "Breakfast", "breakfast foods": "Breakfast",
    "alcohol": "Alcohol", "alcoholic beverages": "Alcohol",
    "snacks": "Snacks",
    "health": "Health", "health & wellness": "Health",
}

def normalize_store_id(values):
    codes = values.astype("string").str.upper().str.extract(r"(\d{3})", expand=False)
    return "FL-" + codes

def clean_transactions(raw):
    df = raw.copy()
    log = []
    def note(issue, decision, rows, reason):
        log.append({"file": "walmart_transactions.csv", "issue": issue, "decision": decision,
                    "rows_affected": int(rows), "reason": reason})

    text_cols = ["transaction_id", "store_id", "timestamp", "sku", "product_name", "category", "unit_price"]
    df[text_cols] = df[text_cols].replace(MISSING_TOKENS, pd.NA)
    missing_cells = int(df[text_cols].isna().sum().sum())
    note("Missing values (blank, NULL, N/A, NaN) - all of them are in timestamp", "Converted to real missing values; handled per column below", missing_cells,
         "So they can be counted and handled instead of read as strings")

    dupes = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    note("Exact duplicate rows", "Dropped", dupes, "A repeated POS record is not a second sale")

    prices = df["unit_price"].astype("string")
    with_dollar = int(prices.str.contains("$", regex=False, na=False).sum())
    df["unit_price"] = pd.to_numeric(prices.str.replace(r"[$,]", "", regex=True).str.strip(), errors="coerce")
    assert df["unit_price"].notna().all(), "some prices did not convert"
    note("unit_price stored as text, some with a $", "Stripped $ and commas, converted to number", with_dollar,
         "Prices need to be numeric to compute revenue")

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    negative = int((df["quantity"] < 0).sum())
    df["is_return"] = df["quantity"] < 0
    note("Negative quantity", "Kept the rows, flagged is_return, left out of demand counts", negative,
         "Negative sales are not demand; kept so unit totals still reconcile to the raw file")

    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    no_ts = int(df["timestamp"].isna().sum())
    df = df.dropna(subset=["timestamp"]).copy()
    df["date"] = df["timestamp"].dt.normalize()
    assert df["date"].between("2024-09-01", "2024-09-30").all(), "a date landed outside September: check day-first formats"
    note("Mixed timestamp formats, some missing", "Parsed all formats (month-first); dropped rows with no timestamp", no_ts,
         "A row with no date cannot be placed in a daily window. Every parsed date is in Sept 2024")

    before = df["store_id"].copy()
    df["store_id"] = normalize_store_id(df["store_id"])
    fixed = int((before.astype("string") != df["store_id"]).sum())
    note("Store ids in several spellings (fl-412, FL_105, ...)", "Standardized to FL-###", fixed,
         "So transactions join to the store file")

    before = df["category"].copy()
    df["category"] = df["category"].astype("string").str.strip().str.lower().map(CATEGORY_MAP)
    assert df["category"].notna().all(), "a category did not map"
    assert df["category"].nunique() == 10, "expected 10 canonical categories"
    changed = int((before.astype("string") != df["category"]).sum())
    note("37 spellings of the same 10 categories", "Mapped to 10 canonical categories", changed,
         "Category names must match for any category comparison")
    return df.reset_index(drop=True), pd.DataFrame(log)

def clean_stores(raw):
    df = raw.copy()
    log = []
    df["store_id"] = normalize_store_id(df["store_id"])
    states = {"Florida": "FL", "Fla.": "FL"}
    state_fix = int(df["state"].isin(states).sum())
    df["state"] = df["state"].replace(states)
    n0 = len(df)
    df = df.drop_duplicates(subset="store_id").copy()
    assert df["store_id"].is_unique
    log.append({"file": "store_locations.csv", "issue": "Same store under two ids (FL239, FL_287 ...)",
                "decision": "Standardized ids, kept one row per store", "rows_affected": n0 - len(df),
                "reason": "A duplicated key would multiply transaction rows in any join"})
    log.append({"file": "store_locations.csv", "issue": "State written as Florida / Fla.",
                "decision": "Standardized to FL", "rows_affected": state_fix, "reason": "Consistent grouping"})
    return df.reset_index(drop=True), pd.DataFrame(log)

clean_tx, tx_log = clean_transactions(tx_raw)
stores, store_log = clean_stores(stores_raw)
decision_log = pd.concat([tx_log, store_log], ignore_index=True)

print("Transactions: rows before cleaning:", len(tx_raw), "| rows after:", len(clean_tx),
      "| removed:", len(tx_raw) - len(clean_tx))
print("Stores:       rows before cleaning:", len(stores_raw), "| rows after:", len(stores))
print("Transaction rows with no matching store:", int((~clean_tx["store_id"].isin(stores["store_id"])).sum()))
print()
print("Decision log (every cleaning decision, with rows touched):")
display(decision_log)

In [ ]:
# Decision-log helper, so cleaning decisions made later (Q4, Q5) land in the same log.
def log_decision(file, issue, decision, rows, reason):
    global decision_log
    decision_log = pd.concat([decision_log, pd.DataFrame([{
        "file": file, "issue": issue, "decision": decision, "rows_affected": int(rows), "reason": reason}])],
        ignore_index=True)

# ---- product_catalog.csv
catalog = pd.read_csv(DATA / "product_catalog.csv")
cat_before = catalog["category"].copy()
catalog["category"] = catalog["category"].str.strip().str.lower().map(CATEGORY_MAP)
assert catalog["category"].notna().all(), "a catalog category did not map"
log_decision("product_catalog.csv", "Category spelled several ways (Emergency / EMERGENCY / Emergency Supplies ...)",
             "Mapped to the same 10 canonical categories as the transactions",
             (cat_before != catalog["category"]).sum(), "Catalog and sales must use one category vocabulary")
missing_cost = int(catalog["unit_cost"].isna().sum())
log_decision("product_catalog.csv", "unit_cost missing", "Left missing, not imputed", missing_cost,
             "unit_cost is not used by any of the five questions; a made-up cost would be fabricated data")

# ---- distribution_centers.csv
dcs = pd.read_csv(DATA / "distribution_centers.csv")
bad_cap = dcs["surge_capacity"].astype("string").str.contains("~", na=False)
dcs["surge_capacity"] = pd.to_numeric(dcs["surge_capacity"].astype("string").str.replace("~", "", regex=False), errors="coerce")
assert dcs["surge_capacity"].notna().all()
log_decision("distribution_centers.csv", "surge_capacity stored as text, one value written '~750'",
             "Stripped the '~' and converted to number", bad_cap.sum(), "Capacity must be numeric to compare centers; the value is approximate in the source")
log_decision("distribution_centers.csv", "cost_per_shipment missing for DC-BHM", "Left missing, not imputed",
             int(dcs["cost_per_shipment"].isna().sum()), "Not used by Q1-Q5; DC-BHM is also the farthest center (503 mi)")

print("Cleaned catalog:", catalog.shape, "| distribution centers:", dcs.shape)
print("Catalog canonical categories:", sorted(catalog["category"].unique()))

## 2. Question 1: Demand surge identification

> Which product categories experienced the greatest sales surge in the 7 days before hurricane landfall compared with the prior baseline period? Quantify the percentage increase and visualize the daily trend for the top 5 categories.

Landfall is Sept 26, 2024. The surge window is the 7 days before it (Sept 19-25). The baseline is Sept 1-15, the flat days before orders first rise on Sept 16, so it represents normal demand.

In [ ]:
# Q1 numbers. Windows are stated explicitly because the multiple depends on them.
BASE_START, BASE_END = "2024-09-01", "2024-09-15"     # flat days before the first rise
SURGE_START, SURGE_END = "2024-09-19", "2024-09-25"   # the 7 days before landfall (landfall = Sept 26)

demand = clean_tx[~clean_tx["is_return"]]             # returns are not demand
daily = (demand.groupby("date").size().rename("orders").to_frame()
         .join(demand.groupby("date")["quantity"].sum().rename("units")))
daily = daily.reindex(pd.date_range("2024-09-01", "2024-09-30", name="date"))
assert daily.notna().all().all(), "a day has no sales"

baseline_per_day = daily.loc[BASE_START:BASE_END, "orders"].mean()
surge_per_day = daily.loc[SURGE_START:SURGE_END, "orders"].mean()
q1_multiple = surge_per_day / baseline_per_day

print(f"Baseline window: {BASE_START} to {BASE_END}  ({len(daily.loc[BASE_START:BASE_END])} days)")
print(f"Surge window:    {SURGE_START} to {SURGE_END}  ({len(daily.loc[SURGE_START:SURGE_END])} days, the 7 days before landfall)")
print(f"Baseline average: {baseline_per_day:,.1f} orders/day")
print(f"Surge average:    {surge_per_day:,.1f} orders/day")
print(f"Multiple:         {q1_multiple:.2f}x")

print()
print("Same data, other defensible windows (so the multiple is never quoted without its windows):")
alternatives = {
    "baseline Sep 1-15, surge Sep 19-25 (ours)": (BASE_START, BASE_END, SURGE_START, SURGE_END),
    "baseline Sep 8-15, surge Sep 19-25":        ("2024-09-08", "2024-09-15", SURGE_START, SURGE_END),
    "baseline Sep 1-18 (runs into the first rise)": ("2024-09-01", "2024-09-18", SURGE_START, SURGE_END),
    "baseline Sep 1-15, surge Sep 23-25 (peak 3 days)": (BASE_START, BASE_END, "2024-09-23", "2024-09-25"),
    "baseline Sep 1-15, surge Sep 20-26 (includes landfall day)": (BASE_START, BASE_END, "2024-09-20", "2024-09-26"),
}
q1_alt = {}
for name, (b0, b1, s0, s1) in alternatives.items():
    q1_alt[name] = daily.loc[s0:s1, "orders"].mean() / daily.loc[b0:b1, "orders"].mean()
    print(f"  {q1_alt[name]:.2f}x  {name}")

# By category: relative (percent) increase, not absolute, in units per day.
cat_daily = (demand.groupby(["date", "category"])["quantity"].sum().unstack("category")
             .reindex(daily.index).fillna(0))
base_c = cat_daily.loc[BASE_START:BASE_END].mean()
surge_c = cat_daily.loc[SURGE_START:SURGE_END].mean()
cat_table = pd.DataFrame({"baseline_units_per_day": base_c, "surge_units_per_day": surge_c})
cat_table["pct_increase"] = (cat_table["surge_units_per_day"] / cat_table["baseline_units_per_day"] - 1) * 100
cat_table["abs_increase"] = cat_table["surge_units_per_day"] - cat_table["baseline_units_per_day"]
cat_table = cat_table.sort_values("pct_increase", ascending=False)
print()
print("Categories ranked by percent increase in units/day (baseline vs surge window):")
display(cat_table.round(1))
top5 = cat_table.index[:5].tolist()
print("Top 5 by percent increase:", top5)
print("Top 5 by absolute increase:", cat_table.sort_values("abs_increase", ascending=False).index[:5].tolist())

In [ ]:
# Q1 chart 1: overall daily orders with the two windows shaded.
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(daily.index, daily["orders"], marker="o", ms=3, color="#24323E")
ax.axvspan(pd.Timestamp(BASE_START), pd.Timestamp(BASE_END), color="#9AA5B1", alpha=0.25, label="Baseline window")
ax.axvspan(pd.Timestamp(SURGE_START), pd.Timestamp(SURGE_END), color="#F67132", alpha=0.20, label="Surge window (7 days before landfall)")
ax.axvline(LANDFALL, color="#F67132", ls="--", label="Landfall (Sept 26)")
ax.set_xlabel("Date (2024)")
ax.set_ylabel("Orders per day (returns excluded)")
landfall_drop = (1 - daily.loc["2024-09-26", "orders"] / daily.loc["2024-09-25", "orders"]) * 100
title = f"Orders ran {q1_multiple:.1f}x normal in the 7 days before landfall, then fell {landfall_drop:.0f}% on landfall day"
ax.set_title(title, loc="left", fontsize=12)
ax.legend(loc="upper left", frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
message_charts.append(title)

# Q1 chart 2: the top 5 categories, daily units.
fig, ax = plt.subplots(figsize=(10, 4.5))
for cat in top5:
    ax.plot(cat_daily.index, cat_daily[cat], marker="o", ms=3, label=f"{cat} (+{cat_table.loc[cat, 'pct_increase']:.0f}%)")
ax.axvline(LANDFALL, color="gray", ls="--", lw=1)
ax.axvspan(pd.Timestamp(SURGE_START), pd.Timestamp(SURGE_END), color="#F67132", alpha=0.10)
ax.set_xlabel("Date (2024)")
ax.set_ylabel("Units sold per day")
title2 = f"{top5[0]} rose the most in relative terms (+{cat_table.loc[top5[0], 'pct_increase']:.0f}%) before landfall"
ax.set_title(title2, loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9, title="Category (% increase vs baseline)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
message_charts.append(title2)

**Q1 answer, in one sentence a manager could read:** In the 7 days before Helene's landfall (Sept 19-25), Florida stores averaged 1,361 orders a day against 576 a day in the Sept 1-15 baseline, 2.4 times normal, and Breakfast (entirely Pop-Tarts in this data) rose the most in percentage terms (+414%) while Emergency supplies added the most units (+652 a day, +184%).

*Windows and caveats.* The baseline is Sept 1-15, the flat days before orders first rise on Sept 16; the surge is Sept 19-25, the 7 days before the Sept 26 landfall, as the brief specifies. The multiple depends on that choice: the five window pairs tried above give 2.2x to 3.1x (the 3.1x uses only the three peak days), so it is never quoted without its windows. Demand excludes returns (negative quantities). Categories are ranked by relative (percent) increase; ranked by absolute units added, Emergency is first and Breakfast second. The top five by percent increase are Breakfast, Emergency, Electronics, Alcohol and Entertainment. Orders fell 43% on landfall day itself, so the demand peak is Sept 24-25, one to two days before landfall.

## 3. Question 2: The Pop-Tarts effect (SKU consolidation)

> After standardizing all Pop-Tart SKU variants into a single product, what is the true daily sales volume? How does the uncleaned (fragmented) view compare with the cleaned (consolidated) view? What business decisions would differ?

The mapping is a table (`sku_map`), not a hard-coded dictionary. A left join lists any SKU that has no mapping, and an assertion proves consolidation does not change unit totals.

In [ ]:
# SKU mapping TABLE (one row per raw SKU). Built by reading the catalog: same product name, size and category.
families = {
    "Pop-Tarts Strawberry":   ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
    "Bottled Water 24pk":     ["WTR-24", "WATER24PK", "WTR_24PK"],
    "AA Batteries 4pk":       ["BATT-100", "BAT-AA4", "BATT_AA_4PK"],
    "LED Flashlight":         ["FLASH-01", "LED-FLASH"],
    "Light Beer 24pk":        ["BEER-24", "LTBEER24"],
    "Monopoly Board Game":    ["BG-MONO", "MONOPOLY-01", "GAME-MONO"],
    "Playing Cards":          ["CARDS-01", "PLAYCARD-STD"],
    "Roku Streaming Device":  ["STREAM-01", "ROKU-STD"],
    "Canned Soup":            ["SOUP-CAN", "CNSOUP-01"],
    "White Bread":            ["BREAD-WH", "WHITEBREAD"],
    "Peanut Butter":          ["PB-16", "PNUTBTR-16"],
    "Emergency Candles":      ["CNDL-EM", "CANDLE-EMRG"],
    "First Aid Kit":          ["FAID-01", "FIRSTAID-KIT"],
    "Portable Charger":       ["CHRG-PORT", "PORTCHARGER"],
    "Chips Variety Pack":     ["CHIPS-VP", "CHIPVARIETY"],
    "Paper Towels 6pk":       ["PTOWEL-6", "PAPERTOWEL6"],
    "Toilet Paper 12pk":      ["TP-12", "TOILETPAPER12"],
    "Baby Diapers Size 3":    ["DIAP-S3", "DIAPERS-3"],
    "Baby Formula":           ["BFORM-01", "BABYFORMULA"],
    "Heavy Duty Tarp":        ["TARP-HD", "HDTARP-01"],
}
sku_map = pd.DataFrame([(sku, canon) for canon, skus in families.items() for sku in skus],
                       columns=["raw_sku", "canonical_product"])
assert sku_map["raw_sku"].is_unique

# Left join catalog -> mapping: anything unmapped shows up as a missing canonical_product.
check = catalog.merge(sku_map, left_on="sku", right_on="raw_sku", how="left", validate="one_to_one")
unmapped = check.loc[check["canonical_product"].isna(), "sku"].tolist()
print("SKUs in the catalog with no mapping:", unmapped)
assert not unmapped

# Apply to the transactions. many_to_one + row-count + unit-total asserts prove nothing was created or lost.
tx2 = clean_tx.merge(sku_map, left_on="sku", right_on="raw_sku", how="left", validate="many_to_one")
assert len(tx2) == len(clean_tx), "mapping join changed the row count"
assert tx2["canonical_product"].notna().all(), "a transaction SKU is missing from the mapping"
assert tx2["quantity"].sum() == clean_tx["quantity"].sum(), "consolidation changed unit totals"
print("Assertion passed: consolidation did not change unit totals "
      f"({int(clean_tx['quantity'].sum()):,} units before and after).")

q2_start_skus = catalog["sku"].nunique()
q2_canonical = sku_map["canonical_product"].nunique()
family_sizes = sku_map.groupby("canonical_product")["raw_sku"].nunique().sort_values(ascending=False)
largest_family = family_sizes.index[0]
print()
print(f"Distinct SKUs at the start: {q2_start_skus}")
print(f"Canonical products at the end: {q2_canonical}")
print(f"Largest consolidated family: {largest_family} ({family_sizes.iloc[0]} SKUs)")
print()
print("Mapping for the largest family:")
display(check.loc[check["canonical_product"] == largest_family, ["sku", "product_name", "package_size", "canonical_product"]])

# Before vs after, for the largest family (units per day, returns excluded).
d2 = tx2[~tx2["is_return"]]
pt = d2[d2["canonical_product"] == largest_family]
frag = pt.groupby(["date", "sku"])["quantity"].sum().unstack("sku").reindex(daily.index).fillna(0)
cons = frag.sum(axis=1)
assert cons.sum() == pt["quantity"].sum()

def mult(series):
    return series.loc[SURGE_START:SURGE_END].mean() / series.loc[BASE_START:BASE_END].mean()

frag_totals = frag.sum().sort_values(ascending=False)
print()
print(f"{largest_family}: units per day, baseline vs surge")
summary = pd.DataFrame({
    "baseline_per_day": [frag[c].loc[BASE_START:BASE_END].mean() for c in frag.columns] + [cons.loc[BASE_START:BASE_END].mean()],
    "surge_per_day": [frag[c].loc[SURGE_START:SURGE_END].mean() for c in frag.columns] + [cons.loc[SURGE_START:SURGE_END].mean()],
    "month_total": list(frag.sum().values) + [cons.sum()],
}, index=list(frag.columns) + ["CONSOLIDATED"])
summary["multiple"] = summary["surge_per_day"] / summary["baseline_per_day"]
display(summary.round(2))

# How the product ranks in the fragmented view vs the consolidated view (month units).
sku_rank = d2.groupby("sku")["quantity"].sum().rank(ascending=False, method="min")
prod_rank = d2.groupby("canonical_product")["quantity"].sum().rank(ascending=False, method="min")
best_frag_sku = frag_totals.index[0]
q2_rank_fragmented = int(sku_rank[best_frag_sku])
q2_rank_consolidated = int(prod_rank[largest_family])
q2_cons_total = int(cons.sum())
q2_best_frag_total = int(frag_totals.iloc[0])
q2_cons_per_day_surge = cons.loc[SURGE_START:SURGE_END].mean()
q2_best_frag_per_day_surge = frag[best_frag_sku].loc[SURGE_START:SURGE_END].mean()
print()
print(f"Best single {largest_family} SKU ({best_frag_sku}): {q2_best_frag_total:,} units, rank #{q2_rank_fragmented} of {d2['sku'].nunique()} SKUs")
print(f"Consolidated {largest_family}: {q2_cons_total:,} units, rank #{q2_rank_consolidated} of {d2['canonical_product'].nunique()} products")
print(f"Consolidated is {q2_cons_total / q2_best_frag_total:.1f}x the best single SKU")
print("Top 3 products by units, consolidated view:", d2.groupby("canonical_product")["quantity"].sum().sort_values(ascending=False).head(3).to_dict())
print("Top 3 SKUs by units, fragmented view:", d2.groupby("sku")["quantity"].sum().sort_values(ascending=False).head(3).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for c in frag.columns:
    axes[0].plot(frag.index, frag[c], marker="o", ms=2, label=c)
axes[0].set_title(f"As recorded: {len(frag.columns)} SKUs, each looks small", loc="left", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)
axes[1].plot(cons.index, cons.values, marker="o", ms=3, color="#B4553F")
axes[1].set_title(f"Consolidated: one product, {q2_cons_total:,} units in September", loc="left", fontsize=11)
for ax in axes:
    ax.axvline(LANDFALL, color="gray", ls="--", lw=1)
    ax.set_xlabel("Date (2024)")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="x", labelrotation=45)
axes[0].set_ylabel("Units sold per day")
fig.suptitle(f"Consolidating {largest_family}: the best SKU is #{q2_rank_fragmented} of {d2['sku'].nunique()}, the product is #{q2_rank_consolidated} of {d2['canonical_product'].nunique()}", x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()
message_charts.append(f"Consolidating {largest_family}: the best SKU is #{q2_rank_fragmented} of {d2['sku'].nunique()}, the product is #{q2_rank_consolidated} of {d2['canonical_product'].nunique()}")

**Q2 answer:** We started with 45 SKUs and ended with 20 canonical products. The largest consolidated family is Pop-Tarts Strawberry (4 SKUs: PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY). The assertion above confirms consolidation did not change unit totals (47,660 net units before and after), and the left join against the mapping table shows no unmapped SKU.

What changes (returns excluded): as recorded, the best single Pop-Tarts SKU (PT-12) sold 1,332 units and ranks #8 of 45 SKUs, and each of the four looks like an ordinary mid-size item. Consolidated, Pop-Tarts sold 5,005 units, 3.8x the best fragment, ranking #2 of 20 products (just behind Bottled Water at 5,037), and its daily volume went from about 73 units in the baseline window to about 377 in the surge window (5.1x). **Business impact:** a buyer reading the fragmented view sees four unremarkable SKUs and would not prioritize them for storm stock; the consolidated view shows one of the top two products with the largest percentage surge in the chain, which changes what gets ordered, how much, and where it is staged. The fragmented view also makes safety-stock rules per SKU wrong, because each code sees only a quarter of the demand.

## 4. Question 3: Weather correlation analysis

> How do wind speed, precipitation and barometric pressure correlate with daily sales volume across categories? Is there a detectable lag between weather severity escalation and purchasing spikes?

**Weather source:** Open-Meteo historical archive API, `https://archive-api.open-meteo.com/v1/archive` (free, no key). We use one location, Orlando (store FL-239), as the clarifications allow. Daily values are New York calendar days; sales timestamps are local Florida time (also Eastern), so both are reduced to a plain date and joined on it.

In [ ]:
# Weather source: Open-Meteo historical archive API (free, no key). Base URL cited below.
BASE = "https://archive-api.open-meteo.com/v1/archive"
RETRY_STATUSES = {429, 500, 502, 503, 504}
ORLANDO = (28.5383, -81.3792)        # store FL-239 (Orlando), the one location we use for weather

def daily_weather(start, end, lat=ORLANDO[0], lon=ORLANDO[1], tries=3):
    params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
              "daily": "precipitation_sum,wind_speed_10m_max,surface_pressure_mean",
              "timezone": "America/New_York"}      # daily values are calendar days in New York time
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        frame = pd.DataFrame(r.json()["daily"]).rename(columns={"time": "date"})
        frame["date"] = pd.to_datetime(frame["date"])        # string -> datetime on the API side
        frame["source"] = "open-meteo (live)"
        return frame

# Saved copy of the same Open-Meteo data (Orlando, Sept 1-30 2024) from the course's Oct 21 studio notebook.
# It has no pressure column. The source column says "fallback sample" so it is never mistaken for a live call.
FALLBACK = pd.DataFrame({
    "date": pd.date_range("2024-09-01", periods=30),
    "precipitation_sum": [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    "wind_speed_10m_max": [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                           15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                           19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    "surface_pressure_mean": np.nan,
    "source": "fallback sample",
})

try:
    weather = daily_weather("2024-09-01", "2024-09-30")
    weather_is_live = True
    dp = (weather["precipitation_sum"] - FALLBACK["precipitation_sum"]).abs().max()
    dw = (weather["wind_speed_10m_max"] - FALLBACK["wind_speed_10m_max"]).abs().max()
    print(f"Live API call worked. Largest difference from the saved copy: precipitation {dp:.2f} mm, wind {dw:.2f} km/h")
except requests.RequestException as err:
    print("Live API call failed:", type(err).__name__, "-", str(err)[:150])
    weather = FALLBACK.copy()
    weather_is_live = False

print("Base URL:", BASE)
print("Weather source:", weather["source"].iloc[0], "| live:", weather_is_live)
print("Weather timezone: America/New_York calendar days. Sales timestamps are local Florida time (also Eastern),")
print("so both sides are reduced to a plain calendar date (timestamp -> .dt.normalize()) and joined on that.")
display(weather.head(3))

# ---- the validated join: weather onto the daily sales table
sales_days = daily.reset_index()[["date", "orders", "units"]]
rows_before = len(sales_days)
merged = sales_days.merge(weather, on="date", how="left", validate="one_to_one", indicator=True)
rows_after = len(merged)
join_rows_preserved = (rows_after == rows_before) and (merged["orders"].sum() == sales_days["orders"].sum())
assert join_rows_preserved, "the join changed the row count or the orders total"
assert not (merged["_merge"] == "right_only").any()
unmatched_days = merged.loc[merged["_merge"] == "left_only", "date"].dt.date.tolist()
print()
print(f"Rows before join: {rows_before} | rows after join: {rows_after} | weather rows: {len(weather)}")
print("Sales days with no weather:", unmatched_days if unmatched_days else "none")
print("Join check passed: row count and orders total unchanged.")

# ---- correlation, same day and with a lag (weather k days AFTER the orders)
def lag_corr(col, k):
    pair = pd.concat([merged["orders"], merged[col].shift(-k)], axis=1).dropna()
    return pair.iloc[:, 0].corr(pair.iloc[:, 1]), len(pair)

metrics = {"wind_speed_10m_max": "max wind (km/h)", "precipitation_sum": "precipitation (mm)",
           "surface_pressure_mean": "mean pressure (hPa)"}
rows = []
for col, label in metrics.items():
    if merged[col].notna().sum() < 10:
        rows.append({"metric": label, "r_same_day": np.nan, "n_days": int(merged[col].notna().sum())})
        continue
    r0, n0 = lag_corr(col, 0)
    rows.append({"metric": label, "r_same_day": r0, "n_days": n0})
corr_same = pd.DataFrame(rows).set_index("metric")
print()
print("Same-day correlation with daily orders:")
display(corr_same.round(2))

lag_rows = []
for col in ["wind_speed_10m_max", "precipitation_sum"]:
    for k in range(0, 5):
        r, n = lag_corr(col, k)
        lag_rows.append({"weather": metrics[col], "weather_k_days_after_orders": k, "r": r, "n_days": n})
lag_table = pd.DataFrame(lag_rows)
print("Lag table (orders on day t vs weather on day t+k):")
display(lag_table.pivot(index="weather_k_days_after_orders", columns="weather", values="r").round(2))

wind_lags = lag_table[lag_table["weather"] == metrics["wind_speed_10m_max"]]
best = wind_lags.loc[wind_lags["r"].abs().idxmax()]
best_k, best_r, best_n = int(best["weather_k_days_after_orders"]), float(best["r"]), int(best["n_days"])
n_corr_days = int(merged["wind_speed_10m_max"].notna().sum())
print(f"Strongest wind relationship: weather {best_k} day(s) after the orders, r = {best_r:.2f}, over n = {best_n} days (same-day n = {n_corr_days}).")

# Robustness: one store (Orlando FL-239) instead of all 12 stores combined.
orl = (demand[demand["store_id"] == "FL-239"].groupby("date").size()
       .reindex(daily.index).fillna(0).rename("orlando_orders").reset_index())
orl = orl.merge(weather[["date", "wind_speed_10m_max"]], on="date", how="left", validate="one_to_one")
orl_r0 = orl["orlando_orders"].corr(orl["wind_speed_10m_max"])
orl_rk = orl["orlando_orders"].corr(orl["wind_speed_10m_max"].shift(-best_k))
print(f"Orlando store only (FL-239): same-day r = {orl_r0:.2f}, {best_k}-day lag r = {orl_rk:.2f}")

In [ ]:
# Q3 chart: orders against wind, landfall marked.
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(merged["date"], merged["orders"], marker="o", ms=3, color="#24323E", label="Orders per day")
ax.axvline(LANDFALL, color="#F67132", ls="--", label="Landfall (Sept 26)")
ax.set_xlabel("Date (2024)")
ax.set_ylabel("Orders per day")
ax2 = ax.twinx()
ax2.plot(merged["date"], merged["wind_speed_10m_max"], color="#21688D", alpha=0.85, label="Max wind (km/h, Orlando)")
ax2.set_ylabel("Max wind speed (km/h)")
title3 = (f"Orders led the wind by {best_k} day(s): r = {best_r:.2f} at that lag, n = {best_n} days "
          f"({'live API' if weather_is_live else 'saved copy'})")
ax.set_title(title3, loc="left", fontsize=12)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False)
for s in ("top",):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
message_charts.append(title3)

In [ ]:
# Q3, across categories: correlation of each category's daily units with each weather indicator.
cat_units = (demand.groupby(["date", "category"])["quantity"].sum().unstack("category")
             .reindex(daily.index).fillna(0))
wx = weather.set_index("date")
specs = [("wind_speed_10m_max", 0, "wind\nsame day"), ("wind_speed_10m_max", 2, "wind\n2 days later"),
         ("precipitation_sum", 0, "rain\nsame day"), ("surface_pressure_mean", 0, "pressure\nsame day")]
mat = {}
for col, k, label in specs:
    if wx[col].notna().sum() < 10:
        print(f"Skipping {col}: no data in this run (the saved fallback has no pressure column)")
        continue
    shifted = wx[col].shift(-k)
    mat[label] = {c: cat_units[c].corr(shifted) for c in cat_units.columns}
heat = pd.DataFrame(mat).loc[cat_table.index]          # rows ordered by Q1 surge size
print("Correlation of daily units with weather, by category (n = 28-30 days each):")
display(heat.round(2))

wind2 = heat["wind\n2 days later"]
others = heat.drop(columns=["wind\n2 days later", "wind\nsame day"]).abs().max().max()
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels(heat.columns)
ax.set_yticks(range(heat.shape[0])); ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(heat.values[i, j]) > 0.6 else "black")
fig.colorbar(im, ax=ax, label="Pearson r")
weak_names = "rain and pressure" if any(c.startswith("pressure") for c in heat.columns) else "rain"
rel = "weaker" if others < wind2.min() else "also notable"
title_h = (f"Wind two days later tracks every category (r = {wind2.min():.2f} to {wind2.max():.2f}); "
           f"{weak_names} are {rel} (|r| <= {others:.2f})")
ax.set_title(title_h, loc="left", fontsize=10)
ax.set_xlabel("Weather indicator (Orlando)")
ax.set_ylabel("Category (ordered by Q1 surge)")
plt.tight_layout(); plt.show()
message_charts.append(title_h)
print("Weather source for this run:", weather["source"].iloc[0])

In [ ]:
# The Q3 answer, printed from the numbers above so it always matches this run.
wind_same = corr_same.loc["max wind (km/h)", "r_same_day"]
print(f"Q3 answer: using {'live' if weather_is_live else 'saved-copy'} Open-Meteo data for Orlando, daily orders and max wind "
      f"are only weakly related on the same day (r = {wind_same:.2f}, n = {n_corr_days} days) but strongly related when the wind "
      f"is compared with orders {best_k} days earlier (r = {best_r:.2f}, n = {best_n} days).")
weak_same = corr_same.drop(index="max wind (km/h)")["r_same_day"].abs().max()
print(f"That is, shoppers bought about two days before the worst wind arrived. Precipitation (and pressure, when the live call supplies it) "
      f"are only weakly related to orders on the same day (|r| <= {weak_same:.2f}). Caveat: 28-30 daily points around a single storm is a small sample.")

**Q3 answer, with its caveat:** Orders and Orlando's daily maximum wind line up best when the wind is compared with orders two days earlier: r = 0.84 over n = 28 days (0.83 with the saved fallback copy), versus 0.39 on the same day over 30 days. In plain terms, shoppers bought roughly two days before the worst wind arrived, so purchasing leads weather severity rather than following it. The heatmap shows the same two-day pattern in every category (r between roughly 0.65 and 0.85 depending on the category, weakest for Snacks), so it is a chain-wide effect, not one category. Precipitation is weakly negative (r from -0.31 to -0.12 across the lags tried) and mean pressure is weakly negative on the same day (r = -0.26), so neither explains the surge. The join was validated: 30 sales days in, 30 out, no weather days missing, orders total unchanged.

*Caveats.* (1) n is small: 28 to 30 daily points, and both series ramp around one storm, so r overstates how much independent evidence there is. (2) Weather comes from one place (Orlando) and is compared with orders from all 12 stores; the Orlando-store-only check gives r = 0.81 at the same lag. (3) We did not report a correlation over only the surge days, because that rests on 3 to 7 points. (4) The code cell prints whether the run used the live Open-Meteo call or the saved fallback copy (the `Weather source:` line), and the chart title says which.

## 5. Question 4: Store-level geographic patterns

> Do all Florida stores experience the same surge pattern, or do stores closer to the hurricane path show earlier and larger spikes? Identify any outlier stores.

Store locations alone do not say which stores were near the storm, so we use Helene's actual track: NOAA National Hurricane Center best-track positions (storm AL092024), saved in `data/helene_best_track.csv`. Source file: `https://ftp.nhc.noaa.gov/atcf/archive/2024/bal092024.dat.gz`. Landfall was about 10 n mi southwest of Perry, FL at 0310 UTC Sept 27, which is 11:10 pm Eastern on Sept 26.

In [ ]:
track = pd.read_csv(DATA / "helene_best_track.csv", parse_dates=["time_utc"])
track["time_et"] = track["time_utc"].dt.tz_convert("America/New_York")
assert len(track) == 25 and track["time_utc"].is_monotonic_increasing
landfall_pt = track.loc[track["time_utc"] == pd.Timestamp("2024-09-27 03:00", tz="UTC")].iloc[0]
print(f"Track: {len(track)} NHC best-track points, {track['time_et'].min():%b %d %H:%M} to {track['time_et'].max():%b %d %H:%M} Eastern")
print(f"Point nearest landfall: {landfall_pt['time_et']:%b %d %H:%M} Eastern at {landfall_pt['latitude']}N, {abs(landfall_pt['longitude'])}W, "
      f"{int(landfall_pt['max_wind_kt'])} kt (about {int(landfall_pt['max_wind_kt'] * 1.15)} mph)")

# Tallahassee has no coordinates in the store file. It is the store nearest landfall, so fill it, flag it, and log it.
stores["coords_filled"] = stores["latitude"].isna()
fill = {"FL-619": (30.4383, -84.2807)}                 # Tallahassee city-center coordinates
for sid, (la, lo) in fill.items():
    stores.loc[stores["store_id"] == sid, ["latitude", "longitude"]] = [la, lo]
assert stores[["latitude", "longitude"]].notna().all().all()
log_decision("store_locations.csv", "Tallahassee (FL-619) has no latitude/longitude",
             "Filled with Tallahassee city-center coordinates (30.4383, -84.2807) and flagged coords_filled",
             int(stores["coords_filled"].sum()),
             "Needed to measure distance to the storm path; Tallahassee is the store nearest landfall, so dropping it would remove the most informative store")

def haversine_miles(lat, lon, lat2, lon2):
    R = 3958.8
    p = np.pi / 180
    a = np.sin((lat2 - lat) * p / 2) ** 2 + np.cos(lat * p) * np.cos(lat2 * p) * np.sin((lon2 - lon) * p / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Densify the track (60 points between each pair of fixes) so distance is to the path, not just to the nearest fix.
pts = track[["latitude", "longitude"]].to_numpy()
dense = np.vstack([np.column_stack([np.linspace(a[0], b[0], 60), np.linspace(a[1], b[1], 60)])
                   for a, b in zip(pts[:-1], pts[1:])])
stores["miles_from_path"] = [haversine_miles(r.latitude, r.longitude, dense[:, 0], dense[:, 1]).min()
                             for r in stores.itertuples()]

# Per-store surge metrics (orders per day, same windows as Q1).
per_store = (demand.groupby(["date", "store_id"]).size().unstack("store_id")
             .reindex(daily.index).fillna(0))
base_n = per_store.loc[BASE_START:BASE_END].sum()
surge_n = per_store.loc[SURGE_START:SURGE_END].sum()
n_base_days = len(per_store.loc[BASE_START:BASE_END]); n_surge_days = len(per_store.loc[SURGE_START:SURGE_END])
base_pd, surge_pd = base_n / n_base_days, surge_n / n_surge_days
mult = surge_pd / base_pd
se = mult * np.sqrt(1 / base_n + 1 / surge_n)          # approximate standard error, treating order counts as Poisson
index_store = per_store / base_pd                        # each store's daily orders as a multiple of its own baseline
assert all(index_store.loc["2024-09-16":"2024-09-25", s_].ge(2.0).any() for s_ in per_store), "a store never reached 2x"
onset = {s_: index_store.loc["2024-09-16":"2024-09-25", s_].ge(2.0).idxmax() for s_ in per_store}   # first day at 2x its own baseline
peak = per_store.idxmax()

store_tbl = stores.set_index("store_id")[["city", "miles_from_path"]].join(pd.DataFrame({
    "orders": per_store.sum(), "surge_multiple": mult, "std_err": se,
    "z_vs_chain": (mult - q1_multiple) / se,
    "first_day_2x": pd.Series(onset), "peak_day": peak}))
store_tbl["first_day_2x"] = pd.to_datetime(store_tbl["first_day_2x"]).dt.strftime("%b %d")
store_tbl["peak_day"] = pd.to_datetime(store_tbl["peak_day"]).dt.strftime("%b %d")
store_tbl = store_tbl.sort_values("miles_from_path")
print(f"Chain-wide surge multiple (Q1): {q1_multiple:.2f}x. Smallest store has {int(store_tbl['orders'].min()):,} demand rows.")
display(store_tbl.round(2))

raw_ids = stores_raw.assign(store_id=normalize_store_id(stores_raw["store_id"]))[["store_id"]]
inflated = demand.merge(raw_ids, on="store_id", how="left")
print(f"Why the store de-duplication matters: joining demand rows to the UNCLEANED store file would turn {len(demand):,} rows into "
      f"{len(inflated):,} (+{len(inflated) / len(demand) - 1:.0%}); the cleaned file keeps {len(demand):,}.")
assert len(demand.merge(stores[["store_id"]], on="store_id", how="left", validate="many_to_one")) == len(demand)

r_dist = store_tbl["miles_from_path"].corr(store_tbl["surge_multiple"])
rho_dist = store_tbl["miles_from_path"].rank().corr(store_tbl["surge_multiple"].rank())    # Spearman = Pearson on ranks
onset_day_num = pd.to_datetime(store_tbl["first_day_2x"] + " 2024", format="%b %d %Y").dt.dayofyear
r_onset = store_tbl["miles_from_path"].corr(onset_day_num)
rng = np.random.default_rng(2002)
perm = np.array([store_tbl["miles_from_path"].corr(pd.Series(rng.permutation(store_tbl["surge_multiple"].values), index=store_tbl.index))
                 for _ in range(10000)])
p_dist = float((np.abs(perm) >= abs(r_dist)).mean())       # two-sided permutation p-value, n = 12 stores
outliers = store_tbl.index[store_tbl["z_vs_chain"].abs() > 2].tolist()
print(f"Distance from path vs surge multiple: Pearson r = {r_dist:.2f}, Spearman rho = {rho_dist:.2f} (n = {len(store_tbl)} stores), permutation p = {p_dist:.2f}")
print(f"Distance from path vs day the store reached 2x its baseline: r = {r_onset:.2f}")
print(f"Surge multiple range: {store_tbl['surge_multiple'].min():.2f}x to {store_tbl['surge_multiple'].max():.2f}x "
      f"(chain {q1_multiple:.2f}x); typical standard error {store_tbl['std_err'].median():.2f}x")
print("Outliers (more than 2 standard errors from the chain-wide multiple):", outliers if outliers else "none")
print("Stores' first day at 2x baseline:", store_tbl["first_day_2x"].value_counts().to_dict())
print("Stores' peak day:", store_tbl["peak_day"].value_counts().to_dict())

In [ ]:
# Q4 chart 1: surge size against distance from the storm path, bubbles sized by order count, bars = approx. standard error.
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.errorbar(store_tbl["miles_from_path"], store_tbl["surge_multiple"], yerr=store_tbl["std_err"],
            fmt="none", ecolor="#9AA5B1", lw=1, zorder=1)
ax.scatter(store_tbl["miles_from_path"], store_tbl["surge_multiple"], s=store_tbl["orders"] / 8,
           color="#21688D", alpha=0.75, zorder=2)
for sid, r in store_tbl.iterrows():
    ax.annotate(f"{r['city']}" + ("*" if sid in fill else ""), (r["miles_from_path"], r["surge_multiple"]),
                textcoords="offset points", xytext=(7, 5), fontsize=8)
ax.axhline(q1_multiple, color="gray", ls=":", lw=1, label=f"Chain-wide {q1_multiple:.2f}x")
ax.set_xlabel("Miles from Helene's track (NHC best track)\n* Tallahassee coordinates filled from city center; bars = approximate standard error")
ax.set_ylabel("Surge multiple (Sept 19-25 vs Sept 1-15 orders/day)")
if r_dist < -0.3:
    shape = "Stores nearer the path surged more"
elif r_dist > 0.3:
    shape = "Stores nearer the path did not surge more (if anything, less)"
else:
    shape = "Surge size was unrelated to distance from the path"
title_q4a = (f"{shape}: r = {r_dist:.2f}, p = {p_dist:.2f}, n = {len(store_tbl)} stores, all within {store_tbl['surge_multiple'].min():.1f}x-{store_tbl['surge_multiple'].max():.1f}x")
ax.set_title(title_q4a, loc="left", fontsize=10.5)
ax.legend(frameon=False, loc="upper right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
message_charts.append(title_q4a)

# Q4 chart 2: timing. Stores nearest the path (under 125 miles) against the rest, each indexed to its own baseline.
near = store_tbl.index[store_tbl["miles_from_path"] < 125].tolist()
far = [s for s in store_tbl.index if s not in near]
grp = pd.DataFrame({"Nearest the path (< 125 mi)": per_store[near].sum(axis=1) / base_pd[near].sum(),
                    "Farther (>= 125 mi)": per_store[far].sum(axis=1) / base_pd[far].sum()})
fig, ax = plt.subplots(figsize=(9, 4.3))
for c, colr in zip(grp.columns, ["#B4553F", "#4A5D7E"]):
    ax.plot(grp.index, grp[c], marker="o", ms=3, color=colr, label=f"{c}: {len(near) if c.startswith('Nearest') else len(far)} stores")
ax.axvline(LANDFALL, color="gray", ls="--", lw=1)
ax.axhline(1, color="gray", ls=":", lw=1)
ax.set_xlabel("Date (2024)")
ax.set_ylabel("Orders as a multiple of own baseline")
gap = (grp.iloc[:, 0] - grp.iloc[:, 1]).loc[SURGE_START:SURGE_END].abs().max()
phrase = "Near and far stores moved together" if gap < 0.35 else "Near and far stores diverged"
title_q4b = f"{phrase}: the largest daily gap in the surge window was {gap:.2f}x of baseline"
ax.set_title(title_q4b, loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()
message_charts.append(title_q4b)
print("Nearest stores:", [store_tbl.loc[s, "city"] for s in near])

**Q4 answer:** No: the stores did not follow the storm path. Measured against the NHC best track, the five stores nearest it (Tallahassee 40 miles, Gainesville 86, Jacksonville 114, St. Petersburg 115, Tampa 122) surged 2.28x-2.43x, while the farthest, Miami (302 miles), surged 2.50x, the highest of all 12. Across the 12 stores, distance and surge size correlate at r = +0.48, the opposite sign from "closer means bigger", but with 12 stores a permutation test gives p = 0.11, so we read it as no detectable relationship rather than a reversed one. Timing says the same: nine stores first reached twice their own baseline on Sept 21 and three (Tallahassee, Gainesville, Orlando) on Sept 22, so the nearest stores were not earlier; and the near and far groups never differed by more than 0.17x of baseline on any day of the surge window (chart 2).

**Outliers:** none by our test. Each store's surge multiple has a standard error of about 0.12x, and the most distant values from the chain-wide 2.36x are Orlando (2.18x, 1.6 standard errors below) and Miami (2.50x, 1.0 above). Orlando is the one to watch, but the gap is within noise.

*Caveats and what we did not test.* n = 12. Tallahassee's coordinates were missing from the store file and were filled from the city center (logged); it is the store nearest landfall, and moving it a few miles would not change that. Distance is to the nearest point on the track, which ran offshore in the Gulf along Florida's west coast; we did not model coastal versus inland exposure. A plausible reason for the uniform pattern, which we did not test, is that the forecast covered most of the peninsula days ahead, so shoppers across the state reacted to the forecast rather than to local conditions.

## 6. Question 5: Signal vs. artifact, the board games investigation

> The board games / entertainment category shows about +250% sales. Using the SQLite inventory and sales database, is this a genuine demand signal or a data artifact (SKU miscoding, category lumping, inventory reporting lag)? Should Walmart stock extra board games for the next storm?

Plan: (1) look at the real schemas, (2) find where +250% comes from, (3) test whether Entertainment moved differently from everything else, (4) cross-check against the transactions, (5) audit SKUs and categories, (6) audit inventory.

In [ ]:
import sqlite3

con = sqlite3.connect(DATA / "inventory_and_sales.db")
for tbl in ["daily_sales_summary", "inventory_levels"]:
    cols = pd.read_sql(f"PRAGMA table_info({tbl})", con)[["name", "type"]].values.tolist()
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", con)["n"][0]
    print(f"{tbl}: {n:,} rows, columns {cols}")

# Dates come in two formats in daily_sales_summary (2024-09-12 and 20240912). Normalize in SQL, then prove nothing was lost or doubled.
DAY = "CASE WHEN date LIKE '%-%' THEN date ELSE substr(date,1,4)||'-'||substr(date,5,2)||'-'||substr(date,7,2) END"
n_compact = pd.read_sql("SELECT SUM(date NOT LIKE '%-%') AS n FROM daily_sales_summary", con)["n"][0]
keys = pd.read_sql(f"SELECT COUNT(*) AS rows_, COUNT(DISTINCT {DAY} || store_id || category) AS keys_ FROM daily_sales_summary", con)
assert keys["rows_"][0] == keys["keys_"][0] == 3600, "a store-day-category appears twice under two date formats"
print(f"{int(n_compact)} of 3,600 rows use the compact YYYYMMDD date. After normalizing, every (day, store, category) is unique: 3,600 rows = 12 stores x 30 days x 10 categories.")
log_decision("inventory_and_sales.db (daily_sales_summary)", "Dates in two formats (YYYY-MM-DD and YYYYMMDD)",
             "Normalized to YYYY-MM-DD in SQL; asserted 3,600 unique store-day-category keys", n_compact,
             "Unparsed dates would silently drop days from the surge window")

sales_sql = f"""
    SELECT {DAY} AS day, category, SUM(total_units) AS units, SUM(total_revenue) AS revenue
    FROM daily_sales_summary GROUP BY day, category ORDER BY day"""
db = pd.read_sql(sales_sql, con, parse_dates=["day"])
raw_total = pd.read_sql("SELECT SUM(total_units) AS u FROM daily_sales_summary", con)["u"][0]
assert abs(db["units"].sum() - raw_total) < 1e-6, "SQL grouping changed the unit total"
db_cat = db.pivot(index="day", columns="category", values="units")
assert db_cat.shape == (30, 10)
db_all = db_cat.sum(axis=1)

PEAK_START, PEAK_END = "2024-09-24", "2024-09-25"       # the two peak days before landfall
def pct(series, a, b):
    return (series.loc[a:b].mean() / series.loc[BASE_START:BASE_END].mean() - 1) * 100

print()
rows = {}
for name, s in [("Entertainment (database)", db_cat["Entertainment"]), ("All categories (database)", db_all),
                ("Entertainment (transactions)", cat_daily["Entertainment"]), ("All products (transactions)", cat_daily.sum(axis=1))]:
    rows[name] = {"7-day surge window %": pct(s, SURGE_START, SURGE_END),
                  "2 peak days (Sep 24-25) %": pct(s, PEAK_START, PEAK_END),
                  "best single day %": (s.loc[SURGE_START:SURGE_END].max() / s.loc[BASE_START:BASE_END].mean() - 1) * 100}
cmp_tbl = pd.DataFrame(rows).T
print("Increase over the Sept 1-15 baseline, units per day. Where does '+250%' come from?")
display(cmp_tbl.round(0))

iso = pd.read_sql("SELECT date AS day, category, SUM(total_units) AS units FROM daily_sales_summary WHERE date LIKE '%-%' GROUP BY day, category",
                  con, parse_dates=["day"]).pivot(index="day", columns="category", values="units")
print(f"Date-format artifact test: ignoring the {int(n_compact)} compact-date rows would give Entertainment +{pct(iso['Entertainment'], PEAK_START, PEAK_END):.0f}% "
      f"on the peak days instead of +{pct(db_cat['Entertainment'], PEAK_START, PEAK_END):.0f}%, so date format is not what produces the number.")

db_rank = ((db_cat.loc[PEAK_START:PEAK_END].mean() / db_cat.loc[BASE_START:BASE_END].mean() - 1) * 100).sort_values(ascending=False)
print("Database, 2 peak days, % increase by category:", db_rank.round(0).to_dict())
print("Entertainment rank among 10 categories:", list(db_rank.index).index("Entertainment") + 1)
share_db = (db_cat["Entertainment"].loc[BASE_START:BASE_END].sum() / db_all.loc[BASE_START:BASE_END].sum() * 100,
            db_cat["Entertainment"].loc[PEAK_START:PEAK_END].sum() / db_all.loc[PEAK_START:PEAK_END].sum() * 100)
tx_all = cat_daily.sum(axis=1)
share_tx = (cat_daily["Entertainment"].loc[BASE_START:BASE_END].sum() / tx_all.loc[BASE_START:BASE_END].sum() * 100,
            cat_daily["Entertainment"].loc[PEAK_START:PEAK_END].sum() / tx_all.loc[PEAK_START:PEAK_END].sum() * 100)
print(f"Entertainment share of all units, baseline -> peak days: database {share_db[0]:.1f}% -> {share_db[1]:.1f}%, transactions {share_tx[0]:.1f}% -> {share_tx[1]:.1f}%")

scale = db_all.loc[BASE_START:BASE_END].mean() / tx_all.loc[BASE_START:BASE_END].mean()
r_sources = db_cat["Entertainment"].corr(cat_daily["Entertainment"])
print(f"Scale check: the database logs {scale:.1f}x the units per day of the transaction file, so compare indexed shapes, not levels. "
      f"Daily Entertainment in the two sources correlates at r = {r_sources:.2f} (n = 30).")

In [ ]:
# SKU and category audit: what is inside "Entertainment", and is a SKU or category problem inflating it?
ent_skus = catalog.loc[catalog["category"] == "Entertainment", "sku"].tolist()
prod = (d2[d2["category"] == "Entertainment"].groupby(["canonical_product", "date"])["quantity"].sum()
        .unstack("canonical_product").reindex(daily.index).fillna(0))
ent_all = prod.sum(axis=1)
audit = pd.DataFrame({
    "skus": sku_map[sku_map["canonical_product"].isin(prod.columns)].groupby("canonical_product")["raw_sku"].nunique(),
    "month_units": prod.sum(),
    "share_of_entertainment_%": prod.sum() / prod.sum().sum() * 100,
    "surge_window_%": [pct(prod[c], SURGE_START, SURGE_END) for c in prod.columns],
    "2_peak_days_%": [pct(prod[c], PEAK_START, PEAK_END) for c in prod.columns]}, index=prod.columns)
audit.loc["ENTERTAINMENT (category)"] = [audit["skus"].sum(), ent_all.sum(), 100.0, pct(ent_all, SURGE_START, SURGE_END), pct(ent_all, PEAK_START, PEAK_END)]
print("Entertainment SKUs in the catalog:", ent_skus)
print("Inside the category (transactions, returns excluded):")
display(audit.round(1))
board = [c for c in prod.columns if "Monopoly" in c][0]
cards = [c for c in prod.columns if "Cards" in c][0]
print(f"'Board games' (Monopoly, {int(audit.loc[board, 'skus'])} SKUs) are {audit.loc[board, 'share_of_entertainment_%']:.0f}% of Entertainment units; "
      f"the rest is playing cards. Category-level surge hides that: board games {audit.loc[board, '2_peak_days_%']:.0f}% vs cards {audit.loc[cards, '2_peak_days_%']:.0f}% on the peak days.")
fragmented_best = d2[d2["canonical_product"] == board].groupby("sku")["quantity"].sum().max()
print(f"Fragmentation check: the best single Monopoly SKU sold {int(fragmented_best):,} units; consolidated {int(prod[board].sum()):,}.")

# Is the categorization consistent between the transaction file and the catalog?
sku_cat_tx = clean_tx.groupby("sku")["category"].nunique()
mismatch = clean_tx.merge(catalog[["sku", "category"]].rename(columns={"category": "catalog_category"}), on="sku", how="left", validate="many_to_one")
n_mis = int((mismatch["category"] != mismatch["catalog_category"]).sum())
print(f"Transaction rows whose category disagrees with the catalog's category for the same SKU: {n_mis}; SKUs sold under more than one category: {int((sku_cat_tx > 1).sum())}")

In [ ]:
# Inventory audit: did board-game stock deplete harder than everything else, or does the inventory table just lag?
inv = pd.read_sql(f"""SELECT {DAY} AS day, store_id, sku, units_on_hand, units_on_order, reorder_point
                      FROM inventory_levels""", con, parse_dates=["day"])
orphans = sorted(set(inv["sku"]) - set(catalog["sku"]))
orph_rows = int(inv["sku"].isin(orphans).sum())
print(f"Inventory has {inv['sku'].nunique()} SKUs; {len(orphans)} are not in the product catalog: {orphans} ({orph_rows} rows, {orph_rows / len(inv):.0%} of the table)")
log_decision("inventory_and_sales.db (inventory_levels)", "4 SKUs not in the product catalog (LEGACY-PT, MISC-001, OLD-BATT-50, UNKNOWN-99)",
             "Kept in the table but excluded from the SKU-level comparison", orph_rows,
             "They cannot be mapped to a product or category, so they cannot be tied to board games or anything else")
inv_ok = inv[~inv["sku"].isin(orphans)]
rows_per_day = inv.groupby("day").size()
print(f"inventory_levels has {rows_per_day.iloc[0]} rows on every one of {len(rows_per_day)} days ({inv['store_id'].nunique()} stores x {inv.groupby(['day', 'store_id']).size().iloc[0]} SKUs of {inv['sku'].nunique()}): "
      f"it is a rotating partial snapshot, not a full store-by-SKU panel.")
log_decision("inventory_and_sales.db (inventory_levels)", "Each day lists only 12 of 49 SKUs per store (rotating partial snapshot)",
             "Compared average units on hand per reported store-SKU row, not sums", len(inv),
             "Summing would mix real stock changes with changes in which stores/SKUs happened to report")
on_hand = inv_ok.groupby(["day", "sku"])["units_on_hand"].mean().unstack("sku")
total_oh = inv_ok.groupby("day")["units_on_hand"].mean()
drop_day = total_oh.pct_change().idxmin()
drop_pct = (1 - total_oh.loc[drop_day] / total_oh.shift(1).loc[drop_day]) * 100
ratio = on_hand.loc[SURGE_START:SURGE_END].mean() / on_hand.loc[BASE_START:BASE_END].mean()
ent_ratio = ratio[ent_skus]
print(f"Average units on hand per reported row fell {drop_pct:.0f}% in one day ({drop_day:%b %d}) across all SKUs and stayed low through Sept 25, then recovered to about "
      f"{total_oh.loc['2024-09-27':'2024-09-30'].mean() / total_oh.loc[BASE_START:BASE_END].mean():.0%} of baseline.")
print(f"On-hand during the surge window as a share of baseline: all SKUs median {ratio.median():.2f}; entertainment SKUs {ent_ratio.round(2).to_dict()}")
below = inv_ok.assign(low=inv_ok["units_on_hand"] < inv_ok["reorder_point"]).groupby("sku")["low"].mean()
print(f"Share of store-days below reorder point: all SKUs median {below.median():.0%}; entertainment SKUs {below[ent_skus].round(2).to_dict()}")
print("Stock-outs (units_on_hand <= 0) anywhere in the table:", int((inv_ok["units_on_hand"] <= 0).sum()))

In [ ]:
# Q5 charts.
idx = pd.DataFrame({
    "Entertainment (database)": db_cat["Entertainment"] / db_cat["Entertainment"].loc[BASE_START:BASE_END].mean(),
    "All categories (database)": db_all / db_all.loc[BASE_START:BASE_END].mean(),
    "Entertainment (transactions)": cat_daily["Entertainment"] / cat_daily["Entertainment"].loc[BASE_START:BASE_END].mean(),
    "All products (transactions)": tx_all / tx_all.loc[BASE_START:BASE_END].mean()})
fig, ax = plt.subplots(figsize=(10, 4.6))
styles = {"Entertainment (database)": ("#B4553F", "-", 2.2), "All categories (database)": ("#B4553F", "--", 1.4),
          "Entertainment (transactions)": ("#21688D", "-", 2.2), "All products (transactions)": ("#21688D", "--", 1.4)}
for c in idx.columns:
    colr, ls, lw = styles[c]
    ax.plot(idx.index, idx[c], color=colr, ls=ls, lw=lw, label=c)
ax.axhline(1, color="gray", ls=":", lw=1); ax.axvline(LANDFALL, color="gray", ls="--", lw=1)
ax.set_xlabel("Date (2024)"); ax.set_ylabel("Units as a multiple of Sept 1-15 average")
pk = idx.loc[PEAK_START:PEAK_END].mean()
title_q5a = (f"On the peak days Entertainment hit {pk['Entertainment (database)']:.1f}x in the database; "
             f"the whole basket hit {pk['All products (transactions)']:.1f}x in the transactions")
ax.set_title(title_q5a, loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="upper left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()
message_charts.append(title_q5a)

oh_idx = on_hand / on_hand.loc[BASE_START:BASE_END].mean()
fig, ax = plt.subplots(figsize=(10, 4.4))
for s_ in ent_skus:
    ax.plot(oh_idx.index, oh_idx[s_], lw=1, alpha=0.8, label=s_)
ax.plot(oh_idx.index, oh_idx.median(axis=1), color="black", lw=2.4, label="Median of all catalog SKUs")
ax.axhline(1, color="gray", ls=":", lw=1); ax.axvline(LANDFALL, color="gray", ls="--", lw=1)
ax.set_xlabel("Date (2024)"); ax.set_ylabel("Avg units on hand per reported row, as a multiple of baseline")
title_q5b = (f"Stock fell {drop_pct:.0f}% across all SKUs on {drop_day:%b %d}; over Sept 19-25 entertainment SKUs averaged "
             f"{ent_ratio.min():.2f}-{ent_ratio.max():.2f} of baseline vs a median of {ratio.median():.2f}")
ax.set_title(title_q5b, loc="left", fontsize=10.5)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()
message_charts.append(title_q5b)
con.close()

**Q5 answer: the +250% is real in direction, but it is a peak-days number and a blend of two products, so it should not be read as a board-game-only signal.**

1. **Where +250% comes from.** In the database, Entertainment is +145% over the 7-day surge window but +267% on the two peak days (Sept 24-25) and +272% on its best single day. The brief's figure is a peak-days number, not a week-long average. In the transactions the same category is +125% over the 7 days and +232% on the peak days.
2. **Is Entertainment special?** The two sources disagree. In the database it beats the basket (+267% vs +200%, rank 3 of 10 behind Breakfast +453% and Emergency +308%, share of units 9.6% to 11.8%). In the transactions it does not (+232% vs +256% for all products, share 9.2% to 8.6%). They agree on timing: daily Entertainment in the two files correlates at r = 0.96. The database logs 5.4x the units of the transaction file, so we compare indexed shapes, not levels.
3. **Category lumping is real, but it hides board games instead of inflating them.** Entertainment is 46% Monopoly (three SKU codes, 1,938 units; the best single code sold only 672) and 54% playing cards. On the peak days Monopoly is +309% against cards +175%; over the 7 days, +160% against +99%. Board games alone ran above the whole basket (+256% peak days, +146% over 7 days); cards ran below it.
4. **SKU miscoding and date formats do not explain it.** No SKU is sold under more than one category and none disagrees with the catalog. Four inventory-only SKUs (LEGACY-PT, MISC-001, OLD-BATT-50, UNKNOWN-99, 8% of inventory rows) cannot be tied to any category, so they cannot be the source. 218 rows use a compact date format; ignoring them would give +278% instead of +267%.
5. **Inventory reporting lag.** Average on-hand units fell 75% on Sept 21 across all SKUs, the same day demand jumped, so the table is not lagging. Over Sept 19-25 the board-game and card SKUs averaged 0.65-0.92 of baseline against 0.75 for all SKUs; only MONOPOLY-01 is clearly below. No stock-outs appear, but the table is a rotating partial snapshot (12 of 49 SKUs per store per day), so it cannot confirm a shortage either.

**Recommendation:** yes, stock extra Monopoly-type board games for the next storm, but size them like the rest of the basket and behind the leaders. Plan for about 2.6x normal daily demand over the 7 days before landfall and about 4.1x on the last two days (+160% and +309% in the transactions), and plan playing cards at only about 2.0x and 2.8x. Pop-Tarts (Breakfast) and Emergency supplies come first. **Confidence is moderate:** one storm, synthetic data, and the two sources disagree about whether the category beats the basket.

## 7. Recommendations for leadership

1. **Stage storm stock early.** Orders first rose on Sept 16 (about 1.4x normal, 10 days out), reached 2x by Sept 21 and peaked on Sept 24-25. Because purchasing leads the wind by about two days, replenishment has to arrive before the 5-day mark, not at landfall.
2. **Put Pop-Tarts first.** Consolidated, Pop-Tarts is the #2 product and its demand ran 5.1x baseline (Breakfast +414%). Fix the SKU master (45 codes for 20 products) so this stops being invisible.
3. **Emergency supplies add the most units** (+652 a day, +184%), so they set the logistics load.
4. **Board games: yes, but sized like the basket** (about 2.6x over the week, 4.1x on the last two days for Monopoly-type games; cards about 2x). Do not treat the +250% headline as board-game-specific.
5. **Allocate chain-wide, not by distance to the forecast track.** All 12 stores surged 2.2x-2.5x with no outliers, so allocate by normal store volume until a storm shows a real geographic gradient.
6. **Clean the source data.** The same store appeared under two ids, categories had 37 spellings for 10 categories, 884 transactions had no timestamp, and the inventory table lists four unmapped SKUs. Each of these changed or could have changed a number in this analysis.

## 8. Complete decision log

Every cleaning decision across all files, with the rows each one touched.

In [ ]:
pd.set_option("display.max_colwidth", 110)
print("Decisions logged:", len(decision_log))
display(decision_log[["file", "issue", "decision", "rows_affected", "reason"]])
print("Rows: transactions", f"{len(tx_raw):,} -> {len(clean_tx):,}", "| stores", f"{len(stores_raw)} -> {len(stores)}",
      "| catalog", len(catalog), "| distribution centers", len(dcs))

## 9. Reflection

*Each team member answers each prompt. **EDIT (Antony):** replace the placeholder under your name in each prompt, then delete the EDIT marker.*

### 1. Data quality impact
> Describe a specific data quality issue you encountered. How did your cleaning decision change the outcome? What would have happened if you had not caught it?

**Danny:** The biggest problem was the Pop-Tarts being split across four SKU codes. Alone, the best code sold 1,332 units and ranked #8 of 45, so nothing looked special. Once I combined them, Pop-Tarts sold 5,005 units, #2 of 20 products, and ran about 5x their normal rate during the surge. If I hadn't caught that, we'd have told Walmart no product stood out, and missed the biggest jump in the data. The store file also had 14 rows for 12 stores, which would have inflated the order table by 17% (24,521 to 28,584 rows). I added a check so that join can't quietly do that again.

**Antony:** EDIT (Antony): write your paragraph here.

### 2. ETL trade-offs
> Pick one decision and explain: what alternative could you have taken, and how might it have changed your results?

**Danny:** The call I could see someone making differently is the Q1 baseline. I used Sept 1-15 as "normal" because demand was flat then, and Sept 19-25 as the surge, which gave 2.36x. Using all of Sept 1-18 gives 2.23x, and only the three peak days gives 3.08x. The headline moves a lot depending on the window, so I show the windows next to every number. I'd make the same choice again. I also dropped 884 rows with no timestamp (3.5%) because they can't be placed on a day, and I never checked whether they bunch up on certain days.

**Antony:** EDIT (Antony): write your paragraph here.

### 3. Pipeline trust
> The Walmart CDO argued that the biggest risk is the pipeline, not the model. Do you agree? What was the most fragile part of your pipeline?

**Danny:** I agree with the CDO. Nothing in the project crashed. The bad data just kept producing believable answers. The most fragile spots were dates and joins. Pandas can read 06/09/2024 as June 9 instead of Sept 6 without any warning, so I added a check that every date falls in September 2024. The weather join can quietly lose the landfall day if the keys don't match, so I check row counts after it. For any new dataset, the first thing I'd add is a row count before and after every step.

**Antony:** EDIT (Antony): write your paragraph here.

### 4. Business vs. data
> For Question 5, you had to make a recommendation with imperfect data. How did you weigh statistical evidence against business risk? Would you decide differently with more time or data?

**Danny:** The data didn't give a clean answer on board games. The database says Entertainment beat the overall basket at the peak (+267% vs +200%), but the transactions say it didn't (+232% vs +256%). The files agree on timing but not on size. Board games alone ran +309% at the peak, and playing cards only +175%. Overstocking board games is cheap since they don't spoil, and running out during a storm loses sales for good. So I recommend extra stock, sized like the overall basket, and behind Pop-Tarts and emergency supplies. With more time I'd want data from several storms, not just Helene.

**Antony:** EDIT (Antony): write your paragraph here.

### 5. Team collaboration
> How did your team divide the work? What would you do differently with another week? What was the most valuable skill each member contributed?

**Danny:** We used one shared repo with a charter, and the weekly labs were our checkpoints. In practice I ended up doing the cleaning, all five questions, the charts, and the write-up myself. Next time I'd split the work by question from day one and set check-in dates, because a charter alone didn't keep the work moving. The skill I brought was catching problems in messy data before they reached the results, mostly by adding checks like the ones above.

**Antony:** EDIT (Antony): write your paragraph here.

## 10. Final self-check

In [ ]:
reflection_complete = False    # TODO: set to True once BOTH team members have replaced their EDIT lines in section 9

checks = {
    "cleaning ran with assertions (rows after cleaning = 24,648)": len(clean_tx) == 24648,
    "decision log covers all five data files": decision_log["file"].str.split(" ").str[0].nunique() >= 5,
    "SKU consolidation did not change unit totals": bool(tx2["quantity"].sum() == clean_tx["quantity"].sum()),
    "weather call visible, base URL cited": BASE.startswith("https://archive-api.open-meteo.com"),
    "weather source for this run is the live API": bool(weather_is_live),
    "weather join preserved every sales day": bool(join_rows_preserved),
    "storm track loaded from NHC best track (25 points)": len(track) == 25,
    "at least 6 charts, each with a message title": len(message_charts) >= 6,
    "reflection written by both team members": reflection_complete,
}
for label, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", label)
print()
print("charts with message titles:", len(message_charts))
for t in message_charts:
    print("  -", t)
print()
print("ready to submit:", all(checks.values()))